In [ ]:
!pip install -q toml colorful librosa==0.9.2 pystoi mir_eval torch_complex GPUtil joblib pesq gdown huggingface_hub
!pip install -q https://github.com/vBaiCai/python-pesq/archive/master.zip

In [ ]:
import os, shutil

REPO_URL = "https://github.com/RookieJunChen/FullSubNet-plus.git"
WORK = "/kaggle/working/FullSubNet-plus"

if os.path.exists(WORK):
    shutil.rmtree(WORK)

!git clone {REPO_URL} {WORK}

os.makedirs(os.path.join(WORK, "train_data_fsn"), exist_ok=True)
print(os.listdir(WORK))

In [ ]:
%cd /kaggle/working/FullSubNet-plus
import os, gdown

CKPT_GDRIVE_ID = "1UJSt1G0P_aXry-u79LLU_l9tCnNa2u7C"

PRETRAINED_DIR = "/kaggle/working/FullSubNet-plus/pretrained"
os.makedirs(PRETRAINED_DIR, exist_ok=True)
PRETRAINED = os.path.join(PRETRAINED_DIR, "FullSubNet+_EN.tar")

if not os.path.exists(PRETRAINED):
    gdown.download(id=CKPT_GDRIVE_ID, output=PRETRAINED, quiet=False)

print("PRETRAINED =", PRETRAINED, "| tồn tại:", os.path.exists(PRETRAINED))

In [ ]:
import os

ROOT = "/kaggle/input/datasets/khabiphmhng/noise-speech"

assert os.path.exists(ROOT), (
    f"Không thấy đường dẫn {ROOT}. Kiểm tra lại tên dataset đã Add ở mục Input bên phải "
)


for root, dirs, files in os.walk(ROOT):
    if os.path.basename(root).upper() == "TRAIN":
        train_root = root
        break

assert train_root is not None, f"Không tìm thấy thư mục TRAIN bên trong {ROOT}."

print("Đã tìm thấy thư mục TRAIN tại:", train_root)
for root, dirs, files in os.walk(train_root):
    depth = root.replace(train_root, "").count(os.sep)
    if depth <= 3:
        print(root)

clean_dirs, noise_dirs = [], []
for root, dirs, files in os.walk(train_root):
    base = os.path.basename(root).upper()
    if base == "CLEAN":
        clean_dirs.append(root)
    elif base == "NOISE":
        noise_dirs.append(root)

print("\nThư mục CLEAN tìm thấy:", clean_dirs)
print("Thư mục NOISE tìm thấy:", noise_dirs)

assert clean_dirs and noise_dirs, (
    "Không tự động tìm thấy thư mục CLEAN/NOISE bên trong TRAIN. Hãy xem cây thư mục in ở trên "
    "và gán tay clean_dirs / noise_dirs cho đúng đường dẫn thực tế."
)

In [ ]:
%cd /kaggle/working/FullSubNet-plus
import glob, random

random.seed(0)

clean_all = []
for d in clean_dirs:
    clean_all += glob.glob(f"{d}/**/*.wav", recursive=True)

noise_all = []
for d in noise_dirs:
    noise_all += glob.glob(f"{d}/**/*.wav", recursive=True)

random.shuffle(clean_all)
random.shuffle(noise_all)

N_VAL_CLEAN = max(25, int(0.05 * len(clean_all)))
N_VAL_NOISE = max(10, int(0.05 * len(noise_all)))

val_clean_pool = clean_all[:N_VAL_CLEAN]
val_noise_pool = noise_all[:N_VAL_NOISE]
train_clean = clean_all[N_VAL_CLEAN:]
train_noise = noise_all[N_VAL_NOISE:]

os.makedirs("train_data_fsn", exist_ok=True)
with open("train_data_fsn/clean.txt", "w") as f:
    f.write("\n".join(train_clean))
with open("train_data_fsn/noise.txt", "w") as f:
    f.write("\n".join(train_noise))
open("train_data_fsn/rir.txt", "w").close()

print(f"Train: {len(train_clean)} clean, {len(train_noise)} noise")
print(f"Val pool: {len(val_clean_pool)} clean, {len(val_noise_pool)} noise")

!wc -l train_data_fsn/clean.txt train_data_fsn/noise.txt

In [ ]:
import numpy as np
import soundfile as sf
import librosa

SR = 16000
random.seed(0)

val_clean_dir = "/kaggle/working/valset/no_reverb/clean"
val_noisy_dir = "/kaggle/working/valset/no_reverb/noisy"
os.makedirs(val_clean_dir, exist_ok=True)
os.makedirs(val_noisy_dir, exist_ok=True)

for i, cpath in enumerate(val_clean_pool):
    clean, _ = librosa.load(cpath, sr=SR)
    noise, _ = librosa.load(random.choice(val_noise_pool), sr=SR)

    if len(noise) < len(clean):
        noise = np.tile(noise, int(np.ceil(len(clean) / len(noise))))
    noise = noise[:len(clean)]

    clean_rms = np.sqrt(np.mean(clean ** 2) + 1e-8)
    noise_rms = np.sqrt(np.mean(noise ** 2) + 1e-8)
    noisy = clean + noise * (clean_rms / (noise_rms + 1e-8))

    peak = np.max(np.abs(noisy)) + 1e-8
    if peak > 1.0:
        noisy, clean = noisy / peak, clean / peak

    sf.write(f"{val_clean_dir}/clean_fileid_{i}.wav", clean, SR)
    sf.write(f"{val_noisy_dir}/mix_fileid_{i}.wav", noisy, SR)

print(f"Đã tạo {len(val_clean_pool)} cặp validation tại /kaggle/working/valset")

In [ ]:
import toml

SAVE_DIR = "/kaggle/working/logs/FullSubNet_plus_finetune"

cfg_path = "/kaggle/working/FullSubNet-plus/config/train.toml"
cfg = toml.load(cfg_path)

cfg.setdefault("model", {})["path"] = PRETRAINED

cfg["meta"]["save_dir"] = SAVE_DIR
cfg["meta"]["use_amp"] = False  
td = cfg["train_dataset"]["args"]
td["clean_dataset"] = "/kaggle/working/FullSubNet-plus/train_data_fsn/clean.txt"
td["noise_dataset"] = "/kaggle/working/FullSubNet-plus/train_data_fsn/noise.txt"
td["rir_dataset"]   = "/kaggle/working/FullSubNet-plus/train_data_fsn/rir.txt"
td["reverb_proportion"] = 0.0
td["num_workers"] = 2

cfg["train_dataset"]["dataloader"]["batch_size"] = 8
cfg["train_dataset"]["dataloader"]["num_workers"] = 2

cfg["validation_dataset"]["args"]["dataset_dir_list"] = [
    "/kaggle/working/valset/no_reverb"
]

cfg["trainer"]["train"]["epochs"] = 20
cfg["trainer"]["train"]["save_checkpoint_interval"] = 1

print("Optimizer config hiện tại:", cfg.get("optimizer", {}))

with open(cfg_path, "w") as f:
    toml.dump(cfg, f)

print(open(cfg_path).read())

In [ ]:
base_trainer_path = "/kaggle/working/FullSubNet-plus/speech_enhance/audio_zen/trainer/base_trainer.py"

with open(base_trainer_path, "r") as f:
    content = f.read()

old = 'self.model.load_state_dict(model_checkpoint["model"], strict=False)'
new = '(self.model.module if hasattr(self.model, "module") else self.model).load_state_dict(model_checkpoint["model"], strict=False)'

assert old in content, "Không tìm thấy đoạn code cần vá — kiểm tra lại nội dung file"
content = content.replace(old, new)

with open(base_trainer_path, "w") as f:
    f.write(content)

print("Đã vá lỗi DDP prefix trong _preload_model")

In [ ]:
import toml

cfg_path = "/kaggle/working/FullSubNet-plus/config/train.toml"
cfg = toml.load(cfg_path)

cfg["model"]["path"] = "fullsubnet_plus.model.fullsubnet_plus.FullSubNet_Plus"
cfg["preloaded_model_path"] = PRETRAINED

cfg["meta"]["save_dir"] = SAVE_DIR

td = cfg["train_dataset"]["args"]
td["clean_dataset"] = "/kaggle/working/FullSubNet-plus/train_data_fsn/clean.txt"
td["noise_dataset"] = "/kaggle/working/FullSubNet-plus/train_data_fsn/noise.txt"
td["rir_dataset"]   = "/kaggle/working/FullSubNet-plus/train_data_fsn/rir.txt"
td["reverb_proportion"] = 0.0
td["num_workers"] = 2

cfg["train_dataset"]["dataloader"]["batch_size"] = 8
cfg["train_dataset"]["dataloader"]["num_workers"] = 2

cfg["validation_dataset"]["args"]["dataset_dir_list"] = [
    "/kaggle/working/valset/no_reverb"
]

cfg["trainer"]["train"]["epochs"] = 5
cfg["trainer"]["train"]["save_checkpoint_interval"] = 1

print("LR gốc:", cfg["optimizer"]["lr"])
cfg["optimizer"]["lr"] = 1e-4  

with open(cfg_path, "w") as f:
    toml.dump(cfg, f)

print(open(cfg_path).read())

In [ ]:
base_trainer_path = "/kaggle/working/FullSubNet-plus/speech_enhance/audio_zen/trainer/base_trainer.py"

with open(base_trainer_path, "r") as f:
    content = f.read()

old1 = 'model_checkpoint = torch.load(model_path.as_posix(), map_location="cpu")'
new1 = 'model_checkpoint = torch.load(model_path.as_posix(), map_location="cpu", weights_only=False)'
assert old1 in content, "Không tìm thấy dòng cần vá (1) — có thể đã vá rồi"
content = content.replace(old1, new1)

old2 = 'checkpoint = torch.load(latest_model_path.as_posix(), map_location="cpu")'
new2 = 'checkpoint = torch.load(latest_model_path.as_posix(), map_location="cpu", weights_only=False)'
if old2 in content:
    content = content.replace(old2, new2)

with open(base_trainer_path, "w") as f:
    f.write(content)

print("Đã vá weights_only=False cho torch.load")

In [ ]:
trainer_path = "/kaggle/working/FullSubNet-plus/speech_enhance/fullsubnet_plus/trainer/trainer.py"

with open(trainer_path, "r") as f:
    content = f.read()

old1 = """        for noisy, clean in self.train_dataloader:
            self.optimizer.zero_grad()"""
new1 = """        print(f"[DEBUG] Epoch {epoch}: so batch trong train_dataloader = {len(self.train_dataloader)}")
        _debug_step = 0

        for noisy, clean in self.train_dataloader:
            self.optimizer.zero_grad()"""

assert old1 in content, "Không tìm thấy đoạn code cần vá (debug log 1) — kiểm tra lại trainer.py"
content = content.replace(old1, new1, 1)

old2 = """            self.scaler.scale(loss).backward()
            self.scaler.unscale_(self.optimizer)
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.clip_grad_norm_value)
            self.scaler.step(self.optimizer)
            self.scaler.update()

            loss_total += loss.item()"""
new2 = """            self.scaler.scale(loss).backward()
            self.scaler.unscale_(self.optimizer)
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.clip_grad_norm_value)
            _scale_before = self.scaler.get_scale()
            self.scaler.step(self.optimizer)
            self.scaler.update()
            _scale_after = self.scaler.get_scale()

            if _debug_step < 3 or _debug_step % 100 == 0:
                _skipped = _scale_after < _scale_before
                print(f"[DEBUG] step {_debug_step}: loss={loss.item():.6f} | scale truoc={_scale_before} sau={_scale_after} | optimizer.step() bi SKIP do inf/nan? {_skipped}")
            _debug_step += 1

            loss_total += loss.item()"""

assert old2 in content, "Không tìm thấy đoạn code cần vá (debug log 2) — kiểm tra lại trainer.py"
content = content.replace(old2, new2, 1)

with open(trainer_path, "w") as f:
    f.write(content)

print("Đã thêm debug log vào _train_epoch của Trainer_Finetune (in ra số batch, loss, và trạng thái skip step do AMP)")

In [ ]:
%cd /kaggle/working/FullSubNet-plus
import torch
n_gpu = torch.cuda.device_count()
print("Số GPU:", n_gpu)

cmd = (
    f'python -m speech_enhance.tools.train '
    f'-C config/train.toml '
    f'-N {n_gpu} '
    f'-P "{PRETRAINED}"'
)
print(cmd)
!{cmd}

In [ ]:
import os, glob

test_root = None
for root, dirs, files in os.walk(ROOT):
    if os.path.basename(root).upper() == "TEST":
        test_root = root
        break

assert test_root is not None, f"Không tìm thấy thư mục TEST bên trong {ROOT}."

print("Đã tìm thấy thư mục TEST tại:", test_root)
for root, dirs, files in os.walk(test_root):
    depth = root.replace(test_root, "").count(os.sep)
    n_wav = len(glob.glob(os.path.join(root, "*.wav")))
    if depth <= 3:
        print(root, "| số file .wav:", n_wav)

INPUT_DIR = test_root

In [ ]:
%cd /kaggle/working/FullSubNet-plus

CKPT = f"{SAVE_DIR}/train/checkpoints/best_model.tar"
OUTPUT_DIR = "/kaggle/working/enhanced_output"

!python -m speech_enhance.tools.inference \
    -C config/inference.toml \
    -M "{CKPT}" \
    -I "{INPUT_DIR}" \
    -O "{OUTPUT_DIR}"

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/enhanced_output", "zip", "/kaggle/working/enhanced_output")
print("Xong: /kaggle/working/enhanced_output.zip")